# SmolVLM Evaluation on PHLOP Dataset

Zero-shot evaluation (4 prompt scenarios) and fine-tuning (4 difficulty configs).

In [3]:
%pip install transformers accelerate
%pip install peft bitsandbytes
%pip install datasets huggingface_hub
%pip install av Pillow matplotlib
%pip install torchvision num2words

/Users/Marichka/Studing/Uni/Project/Iran/PHLOP/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/Marichka/Studing/Uni/Project/Iran/PHLOP/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/Marichka/Studing/Uni/Project/Iran/PHLOP/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/Marichka/Studing/Uni/Project/Iran/PHLOP/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Configuration

In [1]:
import os
import sys
from pathlib import Path
from huggingface_hub import login

sys.path.insert(0, str(Path(".").resolve()))

from phlop_eval_common import (
    load_phlop_splits, EVAL_OPTIONS,
    FINE_TUNE_CONFIGS, get_val_difficulty_filter,
)
from smol_eval import (
    run_zero_shot_smolvlm,
    run_finetune_smolvlm,
    run_test_comparison_smolvlm,
    SMOLVLM_MODEL_ID,
)

REPO_ID = "zimmari-ai/phlop"
HF_TOKEN = os.environ.get("HF_TOKEN", True)
MAX_SAMPLES = None  # None = evaluate all questions; set to e.g. 20 for quick testing
MAX_STEPS = 50
OUTPUT_DIR = "./smolvlm_checkpoints"

# DRIVE_RESULTS_DIR = "/content/drive/MyDrive/Uni/MS Project/results"
DRIVE_RESULTS_DIR = "results"

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

import shutil

def sync_to_drive(local_path: str, drive_dir: str = DRIVE_RESULTS_DIR):
    """Copy a local file to Google Drive for persistence."""
    dst = os.path.join(drive_dir, os.path.basename(local_path))
    shutil.copy2(local_path, dst)
    print(f"  -> Drive: {dst}")

def sync_results_dir(local_dir: str = "results", drive_dir: str = DRIVE_RESULTS_DIR):
    """Copy all files from local results dir to Google Drive."""
    os.makedirs(drive_dir, exist_ok=True)
    for f in sorted(os.listdir(local_dir)):
        src = os.path.join(local_dir, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(drive_dir, f))
    print(f"Synced {local_dir}/ -> {drive_dir}/")

In [ ]:
HF_TOKEN = '**'

In [3]:
if isinstance(HF_TOKEN, str) and HF_TOKEN.startswith("hf_"):
    login(token=HF_TOKEN)

## Load Dataset from HuggingFace

In [4]:
splits = load_phlop_splits(REPO_ID, token=HF_TOKEN)
for name, ds in splits.items():
    print(f"  {name}: {len(ds)} scenes")

Loading splits ['train', 'validation', 'test'] from zimmari-ai/phlop ...


Resolving data files:   0%|          | 0/320 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/320 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

  train: 16000 scenes
  validation: 2000 scenes
  test: 2000 scenes
  train: 16000 scenes
  validation: 2000 scenes
  test: 2000 scenes


## Part 1: Zero-shot Evaluation

4 prompt scenarios (no_additional_info, taxonomy_only, physics_only, taxonomy_and_physics) evaluated on **both val and test** splits with **both static and moving** camera modes.

In [ ]:
import json

os.makedirs("results", exist_ok=True)

# Run ONE camera mode + ONE split per cell to avoid timeouts.
# Change cam / split_name below, or duplicate this cell for each combo.
cam = "static"          # <-- "static" or "moving"
split_name = "validation"  # <-- "validation" or "test"

print(f"Zero-shot — camera: {cam}, split: {split_name}")
result = run_zero_shot_smolvlm(
    splits,
    camera_mode=cam,
    max_samples=MAX_SAMPLES,
    results_dir="results",
    eval_splits=[split_name],
)

cam_path = f"results/smolvlm_zero_shot_metrics_{cam}_{split_name}.json"
with open(cam_path, "w") as f:
    json.dump(result, f, indent=2, default=str)
sync_to_drive(cam_path)

pred_file = f"results/smolvlm_zero_shot_predictions_{split_name}_{cam}.json"
if os.path.exists(pred_file):
    sync_to_drive(pred_file)
print(f"Done: {cam}/{split_name}")

Zero-shot — camera: static, split: validation


Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

Using device: mps
  Building eval index (camera=static)...


Indexing eval scenes: 100%|█████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:36<00:00, 55.52it/s]


  Eval index: 25436 (scene, question) pairs from 2000 scenes

--- Zero-shot on validation (static): 25436 questions (evaluating 25436) ---


In [ ]:
cam = "static"
split_name = "test"

print(f"Zero-shot — camera: {cam}, split: {split_name}")
result = run_zero_shot_smolvlm(
    splits,
    camera_mode=cam,
    max_samples=MAX_SAMPLES,
    results_dir="results",
    eval_splits=[split_name],
)

cam_path = f"results/smolvlm_zero_shot_metrics_{cam}_{split_name}.json"
with open(cam_path, "w") as f:
    json.dump(result, f, indent=2, default=str)
sync_to_drive(cam_path)

pred_file = f"results/smolvlm_zero_shot_predictions_{split_name}_{cam}.json"
if os.path.exists(pred_file):
    sync_to_drive(pred_file)
print(f"Done: {cam}/{split_name}")

In [ ]:
cam = "moving"
split_name = "validation"

print(f"Zero-shot — camera: {cam}, split: {split_name}")
result = run_zero_shot_smolvlm(
    splits,
    camera_mode=cam,
    max_samples=MAX_SAMPLES,
    results_dir="results",
    eval_splits=[split_name],
)

cam_path = f"results/smolvlm_zero_shot_metrics_{cam}_{split_name}.json"
with open(cam_path, "w") as f:
    json.dump(result, f, indent=2, default=str)
sync_to_drive(cam_path)

pred_file = f"results/smolvlm_zero_shot_predictions_{split_name}_{cam}.json"
if os.path.exists(pred_file):
    sync_to_drive(pred_file)
print(f"Done: {cam}/{split_name}")

In [ ]:
cam = "moving"
split_name = "test"

print(f"Zero-shot — camera: {cam}, split: {split_name}")
result = run_zero_shot_smolvlm(
    splits,
    camera_mode=cam,
    max_samples=MAX_SAMPLES,
    results_dir="results",
    eval_splits=[split_name],
)

cam_path = f"results/smolvlm_zero_shot_metrics_{cam}_{split_name}.json"
with open(cam_path, "w") as f:
    json.dump(result, f, indent=2, default=str)
sync_to_drive(cam_path)

pred_file = f"results/smolvlm_zero_shot_predictions_{split_name}_{cam}.json"
if os.path.exists(pred_file):
    sync_to_drive(pred_file)
print(f"Done: {cam}/{split_name}")

In [ ]:
zero_shot_results = {}
for cam in ["static", "moving"]:
    zero_shot_results[cam] = {}
    for split in ["validation", "test"]:
        path = f"results/smolvlm_zero_shot_metrics_{cam}_{split}.json"
        if os.path.exists(path):
            with open(path) as f:
                data = json.load(f)
            zero_shot_results[cam].update(data)

with open("results/smolvlm_zero_shot_metrics.json", "w") as f:
    json.dump(zero_shot_results, f, indent=2, default=str)
sync_to_drive("results/smolvlm_zero_shot_metrics.json")
print("Merged zero-shot metrics into smolvlm_zero_shot_metrics.json")

### Zero-shot Visualizations

Grouped bar charts showing metrics across splits and camera modes, plus accuracy breakdown by question type.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

camera_modes = ["static", "moving"]

with open("results/smolvlm_zero_shot_metrics.json") as f:
    zs_data = json.load(f)

metric_names = ["answer_accuracy", "physics_signal_accuracy", "taxonomy_f1"]
metric_labels = ["Answer Acc", "Physics Signal Acc", "Taxonomy F1"]

groups = []
group_labels = []
for cam in camera_modes:
    for split in ["validation", "test"]:
        entry = zs_data.get(cam, {}).get(split, {})
        m = entry.get("metrics", entry)
        groups.append([m.get(k, 0) for k in metric_names])
        group_labels.append(f"{cam}/{split}")

groups = np.array(groups)
x = np.arange(len(group_labels))
width = 0.22

fig, ax = plt.subplots(figsize=(10, 5))
for i, label in enumerate(metric_labels):
    ax.bar(x + i * width, groups[:, i], width, label=label)
ax.set_xticks(x + width)
ax.set_xticklabels(group_labels, rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("Zero-shot Metrics by Camera Mode and Split")
ax.legend()
plt.tight_layout()
plt.savefig("results/chart_zero_shot_metrics_overview.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, len(camera_modes), figsize=(7 * len(camera_modes), 6), sharey=True)
if len(camera_modes) == 1:
    axes = [axes]
for ax, cam in zip(axes, camera_modes):
    entry = zs_data.get(cam, {}).get("test", zs_data.get(cam, {}).get("validation", {}))
    m = entry.get("metrics", entry)
    pqt = m.get("per_question_type", {})
    if not pqt:
        ax.set_title(f"{cam} — no per-type data")
        continue
    sorted_types = sorted(pqt.items(), key=lambda kv: -kv[1]["count"])
    names = [t for t, _ in sorted_types]
    accs = [v["accuracy"] for _, v in sorted_types]
    counts = [v["count"] for _, v in sorted_types]
    y = np.arange(len(names))
    bars = ax.barh(y, accs)
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Accuracy")
    ax.set_title(f"Zero-shot Accuracy by Question Type ({cam})")
    for bar, c in zip(bars, counts):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"n={c}", va="center", fontsize=7)
plt.tight_layout()
plt.savefig("results/chart_zero_shot_accuracy_by_question_type.png", dpi=150)
plt.show()

sync_to_drive("results/chart_zero_shot_metrics_overview.png")
sync_to_drive("results/chart_zero_shot_accuracy_by_question_type.png")

## Part 2: Fine-tuning

Training uses whichever camera mode the train split provides per sample. Four difficulty configurations:
1. **easy** — train on easy questions, validate on medium/hard/very_hard
2. **easy_medium** — train on easy+medium, validate on hard/very_hard
3. **hard** — train on hard questions, validate on easy/medium/very_hard
4. **full** — train on all questions

In [ ]:
print("Fine-tuning configurations:")
for name, cfg in FINE_TUNE_CONFIGS.items():
    val_diff = get_val_difficulty_filter(cfg["train_difficulty"])
    print(f"  {name}: train={cfg['train_difficulty']}, val={val_diff}")

saved_dirs = run_finetune_smolvlm(
    splits,
    output_dir=OUTPUT_DIR,
    config_name=None,
    max_steps=MAX_STEPS,
    camera_mode="static",  # train split uses per-sample camera_mode anyway
)
print(f"\nSaved checkpoints: {saved_dirs}")

### Fine-tuning Loss Curves

Training and validation loss over steps for each difficulty configuration, read from Trainer state files.

In [ ]:
fig, (ax_train, ax_val) = plt.subplots(1, 2, figsize=(14, 5))

for cfg_name in FINE_TUNE_CONFIGS:
    state_path = os.path.join(OUTPUT_DIR, cfg_name, "trainer_state.json")
    if not os.path.exists(state_path):
        print(f"  No trainer state for {cfg_name}, skipping")
        continue
    with open(state_path) as f:
        state = json.load(f)
    log_history = state.get("log_history", [])

    train_steps = [e["step"] for e in log_history if "loss" in e]
    train_loss = [e["loss"] for e in log_history if "loss" in e]
    if train_steps:
        ax_train.plot(train_steps, train_loss, marker=".", markersize=4, label=cfg_name)

    eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
    eval_loss = [e["eval_loss"] for e in log_history if "eval_loss" in e]
    if eval_steps:
        ax_val.plot(eval_steps, eval_loss, marker="o", markersize=4, label=cfg_name)

ax_train.set_xlabel("Step")
ax_train.set_ylabel("Training Loss")
ax_train.set_title("Training Loss by Config")
ax_train.legend()
ax_train.grid(True, alpha=0.3)

ax_val.set_xlabel("Step")
ax_val.set_ylabel("Validation Loss")
ax_val.set_title("Validation Loss by Config")
ax_val.legend()
ax_val.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/chart_finetune_loss_curves.png", dpi=150)
plt.show()
sync_to_drive("results/chart_finetune_loss_curves.png")

## Part 3: Test Comparison

Compare base model vs all fine-tuned checkpoints on the **test** split only (validation was used to guide training), for **both static and moving** cameras.

In [ ]:
model_checkpoints = [("base", SMOLVLM_MODEL_ID)]
for config_name in FINE_TUNE_CONFIGS:
    ckpt = os.path.join(OUTPUT_DIR, config_name)
    if os.path.isdir(ckpt):
        model_checkpoints.append((config_name, ckpt))

# Run ONE camera mode per cell. Change cam below or duplicate the cell.
cam = "static"  # <-- "static" or "moving"

print(f"Comparing {len(model_checkpoints)} models — camera: {cam}")
table = run_test_comparison_smolvlm(
    splits,
    model_checkpoints,
    camera_mode=cam,
    results_dir="results",
)

cam_data = {name: split_metrics for name, split_metrics in table}
cam_path = f"results/smolvlm_finetuned_metrics_{cam}.json"
with open(cam_path, "w") as f:
    json.dump({cam: cam_data}, f, indent=2, default=str)
sync_to_drive(cam_path)

for fname in os.listdir("results"):
    if fname.startswith("smolvlm_finetuned_predictions_") and fname.endswith(f"_{cam}.json"):
        sync_to_drive(f"results/{fname}")
print(f"Done: comparison for camera={cam}")

In [ ]:
cam = "moving"

print(f"Comparing {len(model_checkpoints)} models — camera: {cam}")
table = run_test_comparison_smolvlm(
    splits,
    model_checkpoints,
    camera_mode=cam,
    results_dir="results",
)

cam_data = {name: split_metrics for name, split_metrics in table}
cam_path = f"results/smolvlm_finetuned_metrics_{cam}.json"
with open(cam_path, "w") as f:
    json.dump({cam: cam_data}, f, indent=2, default=str)
sync_to_drive(cam_path)

for fname in os.listdir("results"):
    if fname.startswith("smolvlm_finetuned_predictions_") and fname.endswith(f"_{cam}.json"):
        sync_to_drive(f"results/{fname}")
print(f"Done: comparison for camera={cam}")

In [ ]:
all_comparisons = {}
for cam in ["static", "moving"]:
    path = f"results/smolvlm_finetuned_metrics_{cam}.json"
    if os.path.exists(path):
        with open(path) as f:
            all_comparisons.update(json.load(f))

with open("results/smolvlm_finetuned_metrics.json", "w") as f:
    json.dump(all_comparisons, f, indent=2, default=str)
sync_to_drive("results/smolvlm_finetuned_metrics.json")
print("Merged finetuned metrics into smolvlm_finetuned_metrics.json")

### Test Comparison Visualizations

Bar charts comparing base model vs fine-tuned checkpoints on the test split.

In [ ]:
camera_modes = ["static", "moving"]

with open("results/smolvlm_finetuned_metrics.json") as f:
    cmp_data = json.load(f)

model_names = list(next(iter(cmp_data.values())).keys()) if cmp_data else []

metric_names = ["answer_accuracy", "physics_signal_accuracy", "taxonomy_f1"]
metric_labels = ["Answer Acc", "Physics Signal Acc", "Taxonomy F1"]

fig, axes = plt.subplots(1, len(camera_modes), figsize=(7 * len(camera_modes), 5), sharey=True)
if len(camera_modes) == 1:
    axes = [axes]
for ax, cam in zip(axes, camera_modes):
    cam_data = cmp_data.get(cam, {})
    accs = []
    labels = []
    for name in model_names:
        m = cam_data.get(name, {}).get("test", {})
        accs.append(m.get("answer_accuracy", 0))
        labels.append(name)
    x = np.arange(len(labels))
    bars = ax.bar(x, accs)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Answer Accuracy")
    ax.set_ylim(0, 1)
    ax.set_title(f"Test Answer Accuracy ({cam})")
    for bar, val in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig("results/chart_finetuned_answer_accuracy.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, len(camera_modes), figsize=(7 * len(camera_modes), 5), sharey=True)
if len(camera_modes) == 1:
    axes = [axes]
width = 0.22
for ax, cam in zip(axes, camera_modes):
    cam_data = cmp_data.get(cam, {})
    labels = []
    vals = {k: [] for k in metric_names}
    for name in model_names:
        m = cam_data.get(name, {}).get("test", {})
        labels.append(name)
        for k in metric_names:
            vals[k].append(m.get(k, 0))
    x = np.arange(len(labels))
    for i, (k, label) in enumerate(zip(metric_names, metric_labels)):
        ax.bar(x + i * width, vals[k], width, label=label)
    ax.set_xticks(x + width)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.set_title(f"All Metrics on Test ({cam})")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("results/chart_finetuned_all_metrics.png", dpi=150)
plt.show()

sync_to_drive("results/chart_finetuned_answer_accuracy.png")
sync_to_drive("results/chart_finetuned_all_metrics.png")

## Results Summary

In [ ]:
import csv

camera_modes = ["static", "moving"]

# Load from files (works standalone)
with open("results/smolvlm_zero_shot_metrics.json") as f:
    zs_summary = json.load(f)
with open("results/smolvlm_finetuned_metrics.json") as f:
    cmp_summary = json.load(f)

print("=" * 60)
print("  ZERO-SHOT RESULTS")
print("=" * 60)
for cam in camera_modes:
    for split in ["validation", "test"]:
        entry = zs_summary.get(cam, {}).get(split, {})
        m = entry.get("metrics", entry)
        n = entry.get("n_questions", "?")
        print(f"\n  Camera: {cam} | Split: {split} | Questions: {n}")
        print(f"    answer_accuracy:          {m.get('answer_accuracy', 0):.4f}")
        print(f"    physics_signal_accuracy:  {m.get('physics_signal_accuracy', 0):.4f}")
        print(f"    taxonomy_f1:              {m.get('taxonomy_f1', 0):.4f}")

print(f"\n{'=' * 60}")
print("  FINE-TUNED MODEL COMPARISON (test only)")
print("=" * 60)
for cam in camera_modes:
    cam_data = cmp_summary.get(cam, {})
    print(f"\n--- Camera: {cam} ---")
    print(f"{'Model':<20} {'AnswerAcc':>10} {'PhysicsAcc':>10} {'TaxF1':>10}")
    print("-" * 55)
    for model_name, split_metrics in cam_data.items():
        m = split_metrics.get("test", {})
        if m:
            print(f"{model_name:<20} {m.get('answer_accuracy', 0):>10.4f} {m.get('physics_signal_accuracy', 0):>10.4f} {m.get('taxonomy_f1', 0):>10.4f}")

with open("results/smolvlm_all_metrics_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["phase", "camera", "split", "model", "answer_accuracy", "physics_signal_accuracy", "taxonomy_f1"])
    for cam in camera_modes:
        for split in ["validation", "test"]:
            entry = zs_summary.get(cam, {}).get(split, {})
            m = entry.get("metrics", entry)
            writer.writerow(["zero_shot", cam, split, "SmolVLM2-2.2B",
                             f"{m.get('answer_accuracy', 0):.4f}",
                             f"{m.get('physics_signal_accuracy', 0):.4f}",
                             f"{m.get('taxonomy_f1', 0):.4f}"])
    for cam, cam_data in cmp_summary.items():
        for model_name, split_metrics in cam_data.items():
            for split_name, m in split_metrics.items():
                writer.writerow(["fine_tuned", cam, split_name, model_name,
                                 f"{m.get('answer_accuracy', 0):.4f}",
                                 f"{m.get('physics_signal_accuracy', 0):.4f}",
                                 f"{m.get('taxonomy_f1', 0):.4f}"])
print(f"\nSaved: results/smolvlm_all_metrics_summary.csv")
sync_to_drive("results/smolvlm_all_metrics_summary.csv")

sync_results_dir()

print(f"\nAll saved artifacts (local + Google Drive):")
print(f"  Models:  {OUTPUT_DIR}/<easy|easy_medium|hard|full>/")
print(f"  Local:   results/")
print(f"  Drive:   {DRIVE_RESULTS_DIR}/")
print(f"\nLocal files:")
for f in sorted(os.listdir("results")):
    print(f"    {f}")
print(f"\nDrive files:")
for f in sorted(os.listdir(DRIVE_RESULTS_DIR)):
    print(f"    {f}")

### Full Predictions Table

Export a single CSV with every prediction across all configurations: video path, question, options, difficulty, true answer, model prediction, camera mode, split, and model name.

In [ ]:
import glob

columns = [
    "phase", "model", "camera_mode", "split",
    "scene_idx", "qa_idx", "video_path",
    "question", "options", "difficulty", "question_type",
    "true_answer", "prediction",
    "prompt",
]

all_rows = []

# Zero-shot prediction files
for pred_file in sorted(glob.glob("results/smolvlm_zero_shot_predictions_*.json")):
    with open(pred_file) as f:
        preds = json.load(f)
    for r in preds:
        all_rows.append({
            "phase": "zero_shot",
            "model": "SmolVLM2-2.2B",
            "camera_mode": r.get("camera_mode", ""),
            "split": r.get("split", ""),
            "scene_idx": r.get("scene_idx", ""),
            "qa_idx": r.get("qa_idx", ""),
            "video_path": r.get("video_path", ""),
            "question": r.get("question", ""),
            "options": str(r.get("options", "")),
            "difficulty": r.get("difficulty", ""),
            "question_type": r.get("question_type", ""),
            "true_answer": r.get("true_answer", r.get("answer", "")),
            "prediction": r.get("prediction", ""),
            "prompt": r.get("prompt", ""),
        })

# Fine-tuned prediction files
for pred_file in sorted(glob.glob("results/smolvlm_finetuned_predictions_*.json")):
    with open(pred_file) as f:
        preds = json.load(f)
    for r in preds:
        all_rows.append({
            "phase": "fine_tuned",
            "model": r.get("model", os.path.basename(pred_file).split("_")[3]),
            "camera_mode": r.get("camera_mode", ""),
            "split": r.get("split", ""),
            "scene_idx": r.get("scene_idx", ""),
            "qa_idx": r.get("qa_idx", ""),
            "video_path": r.get("video_path", ""),
            "question": r.get("question", ""),
            "options": str(r.get("options", "")),
            "difficulty": r.get("difficulty", ""),
            "question_type": r.get("question_type", ""),
            "true_answer": r.get("true_answer", r.get("answer", "")),
            "prediction": r.get("prediction", ""),
            "prompt": r.get("prompt", ""),
        })

out_path = "results/smolvlm_full_predictions_table.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(all_rows)

print(f"Saved {len(all_rows)} predictions to {out_path}")
print(f"  Columns: {columns}")

# Show breakdown
from collections import Counter
breakdown = Counter((r["phase"], r["model"], r["camera_mode"], r["split"]) for r in all_rows)
print(f"\nBreakdown:")
print(f"  {'Phase':<12} {'Model':<20} {'Camera':<10} {'Split':<12} {'Count':>6}")
print(f"  {'-'*65}")
for (phase, model, cam, split), count in sorted(breakdown.items()):
    print(f"  {phase:<12} {model:<20} {cam:<10} {split:<12} {count:>6}")

sync_to_drive(out_path)